# Module 4 Practice: Introduction to pandas

This notebook is a set of practice questions and challenges for **Module 4 (4a–4e)**. It covers:

| Section | Topic | Course notebook |
|---|---|---|
| B1 | Creating a dataframe; naming rows and columns | `module4a` |
| B2 | Selecting rows and columns (`[]`, `.loc`, `.iloc`) and changing values | `module4b` |
| B3 | Adding rows and columns, reordering, `pd.concat`, `.merge` | `module4c` |
| B4 | Saving and loading (`.to_csv`, `pd.read_csv`, `sep`, `header`, `index_col`) | `module4d` |
| B5 | Exploratory analysis, filtering, `.str` methods, missing data | `module4e` |

### How the notebook is organised

- **Part A: Quick recall.** Predict what a snippet does *before* you run it. Write your prediction in the comment, run the cell, then open the answer.
- **Part B: Coding exercises.** Write code to a spec. Each one is followed by a ✅ **check cell** of `assert` tests. A test that fails raises an `AssertionError` with a message saying what is wrong.
- **Part C: Debugging.** Each cell holds broken code. Find and fix the bug(s) so the check cell passes.
- **Part D: Mini-projects.** Two larger problems that combine everything.

Under every question are collapsible **💡 Hint** and **✅ Solution** sections. Try it properly before you open them.

### Tips

- Work **top to bottom** and run cells one at a time. The check cells fail until you write the code, so *Run All* stops at the first unsolved exercise. That is expected.
- Cells marked with the `raises-exception` tag (Part A and C) are *supposed* to error until you have fixed them.
- If you get stuck, every section that builds on earlier work starts with a 🔁 **safety-net** cell that rebuilds the data you need.
- Some exercises use one or two small things not shown in the course (for example `.copy()`, `how="left"`). The exercise explains them when it does.
- Written for pandas ≥ 2.0 and tested on pandas 3.0.

In [ ]:
import re
import numpy as np
import pandas as pd

print("pandas version:", pd.__version__)

---
# Part A: Quick recall ⚡

For each snippet, **predict the output first** (write it in the `# ✍️ My prediction:` comment), then run the cell and compare.

Run this setup cell first:

In [ ]:
mini_df = pd.DataFrame(
    {
        "age": [25, 40, 18, 83, 55],
        "height": [1.75, 1.79, 1.75, 1.63, 1.68],
        "smoker": [False, True, True, False, False],
    },
    index=["p1", "p2", "p3", "p4", "p5"],
)
mini_df

### A · Creating dataframes (4a)

**A1.** What is the shape of `mini_df`?

In [ ]:
# ✍️ My prediction: 

print(mini_df.shape)

<details>
<summary>👀 <strong>Answer</strong></summary>

`(5, 3)`: 5 rows and 3 columns. `.shape` is always `(rows, columns)`, just like NumPy. The index labels (`p1` to `p5`) are **not** a column, so they don't count.

</details>

**A2.** When you build a dataframe from a dictionary, where do the keys go? What are the column labels and the index here, and what changes after `.T`?

In [ ]:
# ✍️ My prediction: 

d = {"p1": [25, 1.75], "p2": [40, 1.79]}
df = pd.DataFrame(d)
print(df)
print("columns:", list(df.columns), "| index:", list(df.index))
print("after .T -> index:", list(df.T.index))

<details>
<summary>👀 <strong>Answer</strong></summary>

Dictionary **keys become column labels** (`p1`, `p2`) and each list becomes a column. The index is the default `0, 1`. After `.T` (transpose), rows and columns swap, so `p1` and `p2` become the **row labels**. This is why the course transposes the participants dictionary.

</details>

**A3.** What happens here?

In [ ]:
# ✍️ My prediction: 

pd.dataframe({"a": [1, 2]})

<details>
<summary>👀 <strong>Answer</strong></summary>

`AttributeError: module 'pandas' has no attribute 'dataframe'`. Names in Python are **case-sensitive**, so it must be `pd.DataFrame`.

</details>

**A4.** What happens if you give `.columns` the wrong number of labels?

In [ ]:
# ✍️ My prediction: 

mini_df.columns = ["age", "height"]

<details>
<summary>👀 <strong>Answer</strong></summary>

`ValueError: Length mismatch: Expected axis has 3 elements, new values have 2 elements`. The list of labels must be exactly as long as the number of columns (`mini_df.shape[1]`). Because the assignment fails, `mini_df` is unchanged.

</details>

### A · Selecting (4b)

**A5.** What *type* is each result? Why does the difference matter?

In [ ]:
# ✍️ My prediction: 

print(type(mini_df["age"]))
print(type(mini_df[["age"]]))

<details>
<summary>👀 <strong>Answer</strong></summary>

Single brackets with one label return a **`Series`** (one-dimensional). Double brackets, which pass a *list* of labels, return a **`DataFrame`** (two-dimensional, even with one column). It matters because some tools only work on one of them. For example, `.str` methods only exist on a Series (see C7).

</details>

**A6.** Which rows and columns does this select?

In [ ]:
# ✍️ My prediction: 

mini_df.iloc[1:3, 0:2]

<details>
<summary>👀 <strong>Answer</strong></summary>

Rows **`p2`, `p3`** (positions 1 and 2) and columns **`age`, `height`** (positions 0 and 1). As in NumPy and Python lists, `.iloc` slices **exclude the end** position.

</details>

**A7.** 🌶️ *Stretch: not covered explicitly in the course.* Slicing with **labels** in `.loc`: which rows come back?

In [ ]:
# ✍️ My prediction: 

mini_df.loc["p2":"p4", "age"]

<details>
<summary>👀 <strong>Answer</strong></summary>

Rows **`p2`, `p3` and `p4`**. Unlike `.iloc`, a `.loc` label slice **includes** the end label. This is a classic gotcha.

</details>

**A8.** What does this return?

In [ ]:
# ✍️ My prediction: 

mini_df.iloc[-1, 0]

<details>
<summary>👀 <strong>Answer</strong></summary>

`55`. Negative positions count from the end, as in NumPy, so `-1` is the last row (`p5`) and column `0` is `age`.

</details>

### A · Filtering (4e)

**A9.** Which participants are returned?

In [ ]:
# ✍️ My prediction: 

mini_df[(mini_df["age"] > 30) & (mini_df["smoker"] == False)]

<details>
<summary>👀 <strong>Answer</strong></summary>

**`p4` and `p5`**. Each condition gives a True/False Series, and `&` combines them element by element. `p2` is over 30 but is a smoker.

</details>

**A10.** Same idea with the word `and`. What happens?

In [ ]:
# ✍️ My prediction: 

mini_df[(mini_df["age"] > 30) and (mini_df["smoker"] == False)]

<details>
<summary>👀 <strong>Answer</strong></summary>

`ValueError: The truth value of a Series is ambiguous...`. Python's `and`/`or` need a *single* True or False, but each condition is a whole Series of them. Use the **bitwise** operators `&` (and) and `|` (or), with each condition in parentheses.

</details>

**A11.** How many rows does this return, and which ones?

In [ ]:
# ✍️ My prediction: 

mini_df.head(-2)

<details>
<summary>👀 <strong>Answer</strong></summary>

**3 rows (`p1`, `p2`, `p3`)**. A negative number in `.head()` returns everything **except** the last *n* rows.

</details>

**A12.** What number does this print?

In [ ]:
# ✍️ My prediction: 

print((mini_df["age"] > 30).sum())

<details>
<summary>👀 <strong>Answer</strong></summary>

`3` (ages 40, 83 and 55). Comparing gives a Boolean Series, and `True` counts as `1` when summed. This is a quick way to **count** rows that meet a condition. The course uses the same trick with `.isnull().sum()`.

</details>

### A · Combining dataframes (4c)

**A13.** What is the index of `stacked`? What would `stacked.loc[0]` return?

In [ ]:
# ✍️ My prediction: 

a = pd.DataFrame({"x": [1, 2]})
b = pd.DataFrame({"x": [3, 4, 5]})
stacked = pd.concat([a, b], axis=0)
print(stacked.index.tolist())

<details>
<summary>👀 <strong>Answer</strong></summary>

`[0, 1, 0, 1, 2]`. Each dataframe keeps its own index, so labels repeat, and `stacked.loc[0]` returns **two rows** (x = 1 and x = 3). Passing `ignore_index=True` gives a clean `0..4`.

</details>

**A14.** What happens when you concatenate *columns* of different lengths?

In [ ]:
# ✍️ My prediction: 

a = pd.DataFrame({"x": [1, 2]})
c = pd.DataFrame({"y": [10, 20, 30]})
print(pd.concat([a, c], axis=1))

<details>
<summary>👀 <strong>Answer</strong></summary>

You get 3 rows. `x` has **`NaN` in row 2** because `a` had no row with label `2`. `x` also becomes a float column, because `NaN` is a float. `pd.concat` lines rows up by **index label**, not by any ID column.

</details>

### A · Saving and missing data (4d, 4e)

**A15.** What do these print?

In [ ]:
# ✍️ My prediction: 

print(pd.Series([2.0, np.nan, 4.0]).mean())
print(pd.Series(["smokes", "Unknown", None]).isnull().sum())

<details>
<summary>👀 <strong>Answer</strong></summary>

`3.0`, because `.mean()` skips `NaN` by default (`skipna=True`). Then `1`: only `None` counts as missing. The string `"Unknown"` is just text to pandas, which is why the course replaces it with `pd.NA`.

</details>

**A16.** After saving to CSV and reading back, what *type* is the `diseases` value?

In [ ]:
# ✍️ My prediction: 

pd.DataFrame({"diseases": [("Asthma",), ()]}).to_csv("roundtrip.csv", index=False)
back = pd.read_csv("roundtrip.csv")
print(type(back.loc[0, "diseases"]), repr(back.loc[0, "diseases"]))

<details>
<summary>👀 <strong>Answer</strong></summary>

`<class 'str'>`. You get the text `"('Asthma',)"`, not a tuple. CSV/TSV files only store **text**, so Python objects such as tuples and lists come back as strings. Numbers and True/False are parsed back to their types automatically.

</details>

---
# Part B: Coding exercises 🛠️

Write code where it says `# Your code here`, then run the ✅ check cell underneath.

## B1 · Creating a dataframe (4a)

Run this setup cell. It recreates the participant data from Module 2 / 4a. **Note:** participant 1's height was recorded in **cm** by mistake; you'll fix this in B2.

In [ ]:
participants_clinical_information = {
    "participant1": ["female", 25, 175, 70.9, False, ("Asthma", "Diabetes")],
    "participant2": ["male", 40, 1.79, 103.4, True, ()],
    "participant3": ["male", 18, 1.75, 85.1, True, ("Lung cancer",)],
    "participant4": ["female", 83, 1.63, 55.9, False, ("Cardio vascular disease", "Alzheimers")],
    "participant5": ["female", 55, 1.68, 50.0, False, ("Asthma", "Anxiety")],
    "participant6": ["female", 32, 1.59, 64.0, True, ("Diabetes",)],
    "participant7": ["male", 21, 1.90, 92.9, False, ("Asthma", "Colon cancer")],
    "participant8": ["male", 46, 1.71, 75.4, False, ()],
    "participant9": ["female", 32, 1.66, 90.7, True, ("Depression",)],
    "participant10": ["male", 67, 1.78, 82.3, False, ("Anxiety", "Diabetes", "Cardio vascular disease")],
}
COLUMNS = ["gender", "age", "height", "weight", "smoking", "diseases"]

### B1.1 · Dictionary → dataframe with labels

1. Create `clinical_df` from `participants_clinical_information` so that there is **one row per participant**.
2. Name the columns using `COLUMNS`.
3. Make sure the **row labels are the participant names** (`"participant1"`, …), not numbers.
4. Run `print(clinical_df.dtypes)`. Every column is `object`, because transposing a mixed table loses the column types. Fix this with `clinical_df = clinical_df.infer_objects()`, which asks pandas to work out a proper type for each column.

In [ ]:
# Your code here
# clinical_df = 

In [ ]:
# ✅ Run this cell to check B1.1
assert isinstance(clinical_df, pd.DataFrame), "clinical_df should be a DataFrame"
assert clinical_df.shape == (10, 6), f"Expected shape (10, 6), got {clinical_df.shape}. Did you transpose?"
assert list(clinical_df.columns) == COLUMNS, "Columns should be named using COLUMNS"
assert clinical_df.index[0] == "participant1", "Row labels should be the participant names"
assert clinical_df.loc["participant4", "age"] == 83
assert pd.api.types.is_integer_dtype(clinical_df["age"]), "age should be an integer column. Did you use .infer_objects()?"
assert pd.api.types.is_float_dtype(clinical_df["weight"]), "weight should be a float column"
assert pd.api.types.is_bool_dtype(clinical_df["smoking"]), "smoking should be a bool column"
print("✅ B1.1 passed!")

<details>
<summary>💡 <strong>Hint</strong></summary>

Dictionary keys become **columns**, so you need `.T` to turn them into rows. After transposing, the old column labels (the keys) are already the index, so step 3 may need no extra code.

</details>

<details>
<summary>✅ <strong>Solution</strong></summary>

```python
clinical_df = pd.DataFrame(participants_clinical_information).T
clinical_df.columns = COLUMNS
print(clinical_df.dtypes)          # all object
clinical_df = clinical_df.infer_objects()
print(clinical_df.dtypes)          # int, float, bool, ...
clinical_df
```

Unlike the 4a exercise, there is no need to set the index here. Transposing moves the dictionary keys into the index for free.

</details>

### B1.2 · NumPy array → dataframe

1. Use `np.arange` and `.reshape` to make a **4 × 3** array containing the numbers 1 to 12.
2. Turn it into a dataframe called `lab_df` with columns `glucose`, `insulin`, `cortisol`.
3. Label the rows `sample_1` … `sample_4`. **Build the labels with a loop and an f-string, don't type them by hand.**

In [ ]:
# Your code here
# lab_df = 

In [ ]:
# ✅ Run this cell to check B1.2
assert lab_df.shape == (4, 3), f"Expected shape (4, 3), got {lab_df.shape}"
assert list(lab_df.columns) == ["glucose", "insulin", "cortisol"]
assert list(lab_df.index) == ["sample_1", "sample_2", "sample_3", "sample_4"]
assert lab_df.loc["sample_3", "insulin"] == 8, "Values should be 1..12 filled row by row"
assert lab_df.loc["sample_4", "cortisol"] == 12
print("✅ B1.2 passed!")

<details>
<summary>💡 <strong>Hint</strong></summary>

`np.arange(1, 13)` gives 1 to 12, because the end is excluded. `.reshape(4, 3)` gives 4 rows and 3 columns. For the labels, start with an empty list and `.append(f"sample_{i + 1}")` inside `for i in range(...)`.

</details>

<details>
<summary>✅ <strong>Solution</strong></summary>

```python
array = np.arange(1, 13).reshape(4, 3)
lab_df = pd.DataFrame(array)
lab_df.columns = ["glucose", "insulin", "cortisol"]

sample_labels = []
for i in range(lab_df.shape[0]):
    sample_labels.append(f"sample_{i + 1}")
lab_df.index = sample_labels
# (one-liner alternative: [f"sample_{i}" for i in range(1, 5)])
lab_df
```

</details>

### B1.3 · A dictionary of *columns* (no transpose needed)

This time the dictionary keys are the **measurements**, not the people. Create `vitals_df` from:

```python
{"heart_rate": [72, 88, 65], "systolic_bp": [120, 135, 110]}
```

and label the rows `mon`, `tue`, `wed`. Should you transpose this time?

In [ ]:
# Your code here
# vitals_df = 

In [ ]:
# ✅ Run this cell to check B1.3
assert vitals_df.shape == (3, 2), f"Expected shape (3, 2), got {vitals_df.shape}. Did you transpose by mistake?"
assert list(vitals_df.columns) == ["heart_rate", "systolic_bp"]
assert list(vitals_df.index) == ["mon", "tue", "wed"]
assert vitals_df.loc["tue", "systolic_bp"] == 135
print("✅ B1.3 passed!")

<details>
<summary>💡 <strong>Hint</strong></summary>

Keys become columns, and here you *want* the keys as columns.

</details>

<details>
<summary>✅ <strong>Solution</strong></summary>

```python
vitals_df = pd.DataFrame({"heart_rate": [72, 88, 65], "systolic_bp": [120, 135, 110]})
vitals_df.index = ["mon", "tue", "wed"]
vitals_df
```

**No transpose.** Whether you need `.T` depends on what the dictionary keys represent. If the keys should be column headers, use the dictionary directly.

</details>

## B2 · Selecting rows and columns, changing values (4b)

🔁 **Safety net:** run this cell to make sure `clinical_df` is correct before you continue.

In [ ]:
clinical_df = pd.DataFrame(participants_clinical_information).T
clinical_df.columns = COLUMNS
clinical_df = clinical_df.infer_objects()
clinical_df.head(3)

### B2.1 · Selecting columns

1. Store the `age` column as a **Series** in `ages`.
2. Store the `gender` and `smoking` columns as a **DataFrame** in `gender_smoking`.

In [ ]:
# Your code here
# ages = 
# gender_smoking = 

In [ ]:
# ✅ Run this cell to check B2.1
assert isinstance(ages, pd.Series), "ages should be a Series (single brackets)"
assert ages.name == "age"
assert isinstance(gender_smoking, pd.DataFrame), "gender_smoking should be a DataFrame (list of names in brackets)"
assert list(gender_smoking.columns) == ["gender", "smoking"]
print("✅ B2.1 passed!")

<details>
<summary>💡 <strong>Hint</strong></summary>

One column name in brackets gives a Series. A *list* of names in brackets (`[[...]]`) gives a DataFrame.

</details>

<details>
<summary>✅ <strong>Solution</strong></summary>

```python
ages = clinical_df["age"]
gender_smoking = clinical_df[["gender", "smoking"]]
```

</details>

### B2.2 · Selecting by label with `.loc`

1. Store participant 7's weight in `p7_weight`, using **one** `.loc[row, column]` call.
2. Store the `age` and `diseases` of participants **2, 4 and 6** in `subset`, using one `.loc` call with lists for both rows and columns.

In [ ]:
# Your code here
# p7_weight = 
# subset = 

In [ ]:
# ✅ Run this cell to check B2.2
assert p7_weight == 92.9, f"Expected 92.9, got {p7_weight}"
assert subset.shape == (3, 2), f"Expected shape (3, 2), got {subset.shape}"
assert list(subset.index) == ["participant2", "participant4", "participant6"]
assert list(subset.columns) == ["age", "diseases"]
print("✅ B2.2 passed!")

<details>
<summary>💡 <strong>Hint</strong></summary>

`df.loc[row_label, column_label]` for one value, or `df.loc[[rows...], [cols...]]` for a block.

</details>

<details>
<summary>✅ <strong>Solution</strong></summary>

```python
p7_weight = clinical_df.loc["participant7", "weight"]
subset = clinical_df.loc[["participant2", "participant4", "participant6"], ["age", "diseases"]]
subset
```

</details>

### B2.3 · Selecting by position with `.iloc`

1. `block`: participants 4 to 7 **inclusive** with the `height` and `weight` columns (column positions 2 and 3), using a single `.iloc` with slices.
2. `last_row`: the **last** row of the dataframe, using a negative position.
3. `corners`: the values at row positions 0 and 9 and column positions 0 and 5, using `.iloc` with lists.

In [ ]:
# Your code here
# block = 
# last_row = 
# corners = 

In [ ]:
# ✅ Run this cell to check B2.3
assert block.shape == (4, 2), f"Expected shape (4, 2), got {block.shape}. Remember the end position is excluded"
assert list(block.index) == ["participant4", "participant5", "participant6", "participant7"]
assert list(block.columns) == ["height", "weight"]
assert isinstance(last_row, pd.Series) and last_row.name == "participant10", "last_row should be participant10 as a Series"
assert corners.shape == (2, 2) and list(corners.columns) == ["gender", "diseases"]
print("✅ B2.3 passed!")

<details>
<summary>💡 <strong>Hint</strong></summary>

Participant 4 is at **position 3**. To *include* position 6 the slice must end at 7. Height and weight are positions 2 and 3, so use `2:4`.

</details>

<details>
<summary>✅ <strong>Solution</strong></summary>

```python
block = clinical_df.iloc[3:7, 2:4]
last_row = clinical_df.iloc[-1]
corners = clinical_df.iloc[[0, 9], [0, 5]]
block
```

</details>

### B2.4 · Fixing a value

Participant 1's height is `175` (cm), but every other height is in metres. Change it to metres **in `clinical_df`** using a single `.loc[row, column] = ...` assignment.

In [ ]:
# Your code here

In [ ]:
# ✅ Run this cell to check B2.4
assert abs(clinical_df.loc["participant1", "height"] - 1.75) < 1e-9, "participant1's height should now be 1.75"
assert (clinical_df["height"] < 2.5).all(), "All heights should be in metres"
print("✅ B2.4 passed!")

<details>
<summary>💡 <strong>Hint</strong></summary>

Use `clinical_df.loc["participant1", "height"] = ...`, with the row and column *inside the same brackets*. Avoid `clinical_df.loc["participant1"]["height"] = ...`; part C3 explains why.

</details>

<details>
<summary>✅ <strong>Solution</strong></summary>

```python
clinical_df.loc["participant1", "height"] = clinical_df.loc["participant1", "height"] / 100
clinical_df.head(3)
```

Dividing by 100 instead of typing `1.75` means the code still works if the stored value was, say, 176.

</details>

### B2.5 · Changing several values on a copy

You want to test how your analysis copes with missing ages, **without** changing `clinical_df`.

1. Make an independent copy: `clinical_copy = clinical_df.copy()`. (`.copy()` is new. Without it, both names would point to the *same* dataframe.)
2. Using **one** `.loc` assignment, set the `age` of participants 3 and 9 in `clinical_copy` to `np.nan`.

In [ ]:
# Your code here

In [ ]:
# ✅ Run this cell to check B2.5
assert clinical_copy["age"].isnull().sum() == 2, "Exactly 2 ages should be missing in clinical_copy"
assert pd.isnull(clinical_copy.loc["participant3", "age"]) and pd.isnull(clinical_copy.loc["participant9", "age"])
assert clinical_df["age"].isnull().sum() == 0, "clinical_df should be unchanged. Did you use .copy()?"
print("✅ B2.5 passed!")

<details>
<summary>💡 <strong>Hint</strong></summary>

`df.loc[[row1, row2], "age"] = np.nan` sets several cells at once.

</details>

<details>
<summary>✅ <strong>Solution</strong></summary>

```python
clinical_copy = clinical_df.copy()
clinical_copy.loc[["participant3", "participant9"], "age"] = np.nan
clinical_copy
```

Notice that `age` turned from int to float in the copy, because `NaN` is a float value.

</details>

## B3 · Expanding and combining dataframes (4c)

🔁 **Safety net + setup:** run this cell. It rebuilds `clinical_df` with participant 1's height fixed and creates two new tables. Note that `labs_df` uses a **different key name** (`pid`), is in a **different order**, is **missing** participants 5 and 11, and includes a participant (`participant99`) who isn't in our study.

In [ ]:
clinical_df = pd.DataFrame(participants_clinical_information).T
clinical_df.columns = COLUMNS
clinical_df = clinical_df.infer_objects()
clinical_df.loc["participant1", "height"] = 1.75

labs_df = pd.DataFrame({
    "pid": ["participant6", "participant1", "participant99", "participant2", "participant3",
            "participant10", "participant4", "participant7", "participant8", "participant9"],
    "cholesterol": [5.2, 4.8, 6.1, 5.9, 4.4, 6.3, 5.0, 4.9, 5.5, 5.1],
})

extra_df = pd.DataFrame({
    "participant_id": ["participant12", "participant13"],
    "gender": ["female", "male"],
    "age": [29, 61],
    "height": [1.70, 1.84],
    "weight": [61.2, 90.5],
    "smoking": [False, True],
    "diseases": [(), ("Hypertension",)],
})
labs_df

### B3.1 · Add a calculated column

Add a `bmi` column to `clinical_df`, calculated as `weight / height ** 2` for every participant in one line.

In [ ]:
# Your code here

In [ ]:
# ✅ Run this cell to check B3.1
assert "bmi" in clinical_df.columns, "clinical_df should have a 'bmi' column"
assert abs(clinical_df.loc["participant2", "bmi"] - 32.27) < 0.01
assert abs(clinical_df.loc["participant1", "bmi"] - 23.15) < 0.01, "Check participant1's height is 1.75 (run the setup cell)"
print("✅ B3.1 passed!")

<details>
<summary>💡 <strong>Hint</strong></summary>

Operations on columns work element by element, like NumPy arrays: `df["a"] / df["b"] ** 2`.

</details>

<details>
<summary>✅ <strong>Solution</strong></summary>

```python
clinical_df["bmi"] = clinical_df["weight"] / clinical_df["height"] ** 2
clinical_df.head()
```

</details>

### B3.2 · Turn the index into a column, renumber and reorder

Create `cohort_df` as a **copy** of `clinical_df`, then:
1. add a `participant_id` column containing the current row labels;
2. replace the index with `0, 1, 2, …` using `np.arange` and the number of rows from `.shape`;
3. reorder the columns so they are: `participant_id, gender, age, height, weight, smoking, diseases, bmi`.

In [ ]:
# Your code here
# cohort_df = 

In [ ]:
# ✅ Run this cell to check B3.2
assert list(cohort_df.columns) == ["participant_id", "gender", "age", "height", "weight", "smoking", "diseases", "bmi"]
assert list(cohort_df.index) == list(range(10)), "Index should be 0..9"
assert cohort_df.loc[0, "participant_id"] == "participant1"
assert "participant_id" not in clinical_df.columns, "clinical_df should be unchanged. Use .copy()"
print("✅ B3.2 passed!")

<details>
<summary>💡 <strong>Hint</strong></summary>

Reorder by selecting all columns with a list in the order you want: `df = df[[...]]`.

</details>

<details>
<summary>✅ <strong>Solution</strong></summary>

```python
cohort_df = clinical_df.copy()
cohort_df["participant_id"] = cohort_df.index
cohort_df.index = np.arange(0, cohort_df.shape[0])
cohort_df = cohort_df[["participant_id", "gender", "age", "height", "weight", "smoking", "diseases", "bmi"]]
cohort_df
```

</details>

### B3.3 · Append a row

Add **participant11** as a new row with label `10`: female, 34 years, 1.55 m, 64.0 kg, non-smoker, no diseases (`()`), and the correct BMI. Use `.loc` and calculate the BMI in code rather than typing it.

In [ ]:
# Your code here

In [ ]:
# ✅ Run this cell to check B3.3
assert cohort_df.shape == (11, 8), f"Expected 11 rows and 8 columns, got {cohort_df.shape}"
assert cohort_df.loc[10, "participant_id"] == "participant11"
assert abs(cohort_df.loc[10, "bmi"] - 26.64) < 0.01
print("✅ B3.3 passed!")

<details>
<summary>💡 <strong>Hint</strong></summary>

`df.loc[new_label] = [value_for_col1, value_for_col2, ...]`. Values must be in the same order as the columns.

</details>

<details>
<summary>✅ <strong>Solution</strong></summary>

```python
cohort_df.loc[10] = ["participant11", "female", 34, 1.55, 64.0, False, (), 64.0 / 1.55 ** 2]
cohort_df.tail(3)
```

</details>

### B3.4 · Merge with differently named, shuffled keys

1. `inner_df`: merge `cohort_df` with `labs_df`, matching `participant_id` to `pid` (the default merge, as in the course).
2. `left_df`: the same merge, but **keep every participant** in `cohort_df` even if they have no lab result. Add the argument `how="left"`, which is new.
3. Remove the duplicate key column `pid` from `left_df`, either by selecting the columns you want or with `left_df.drop(columns="pid")`.

Before running the check, predict how many rows each result has, and why.

In [ ]:
# Your code here
# inner_df = 
# left_df = 

In [ ]:
# ✅ Run this cell to check B3.4
assert len(inner_df) == 9, f"inner_df should have 9 rows, got {len(inner_df)}"
assert len(left_df) == 11, f"left_df should have 11 rows (all participants), got {len(left_df)}"
assert "pid" not in left_df.columns, "Remove the duplicate 'pid' column"
assert "participant99" not in left_df["participant_id"].values
p1_chol = left_df.loc[left_df["participant_id"] == "participant1", "cholesterol"].iloc[0]
assert p1_chol == 4.8, "participant1 should be matched to cholesterol 4.8"
assert left_df["cholesterol"].isnull().sum() == 2, "participants 5 and 11 should have missing cholesterol"
print("✅ B3.4 passed!")

<details>
<summary>💡 <strong>Hint</strong></summary>

`cohort_df.merge(labs_df, left_on=..., right_on=..., how=...)`. The `left` table is the one you call `.merge` on.

</details>

<details>
<summary>✅ <strong>Solution</strong></summary>

```python
inner_df = cohort_df.merge(labs_df, left_on="participant_id", right_on="pid")
left_df = cohort_df.merge(labs_df, left_on="participant_id", right_on="pid", how="left")
left_df = left_df.drop(columns="pid")
left_df
```

- **Default (inner) merge: 9 rows.** It keeps only IDs found in *both* tables, so participants 5 and 11 (no labs) and participant99 (not in the cohort) are dropped.
- **Left merge: 11 rows.** It keeps every row of the left table, and participants 5 and 11 get `NaN` cholesterol. participant99 still doesn't appear because it isn't in the left table.
- Either way, `.merge` matched the shuffled rows by ID, so the order of `labs_df` didn't matter.

</details>

### B3.5 · Why not just `pd.concat`?

1. Create `naive_df = pd.concat([cohort_df, labs_df], axis=1)`.
2. Using a filter, count how many rows have a `participant_id` that does **not** equal their `pid`. Store the count in `n_mismatched`.

In [ ]:
# Your code here
# naive_df = 
# n_mismatched = 

In [ ]:
# ✅ Run this cell to check B3.5
assert naive_df.shape == (11, 10), f"Expected (11, 10), got {naive_df.shape}"
assert n_mismatched == 11, f"Expected 11 mismatched rows, got {n_mismatched}"
print("✅ B3.5 passed!")

<details>
<summary>💡 <strong>Hint</strong></summary>

`!=` between two columns gives a Boolean Series, and `.sum()` counts the `True` values.

</details>

<details>
<summary>✅ <strong>Solution</strong></summary>

```python
naive_df = pd.concat([cohort_df, labs_df], axis=1)
n_mismatched = (naive_df["participant_id"] != naive_df["pid"]).sum()
print(n_mismatched)
naive_df[["participant_id", "pid", "cholesterol"]]
```

**Every** row is mismatched. `pd.concat(axis=1)` pastes tables side by side by **index position/label**. It knows nothing about IDs. If the tables aren't in exactly the same order, use `.merge`.

</details>

### B3.6 · Concatenate rows

1. Stack `left_df` and `extra_df` (two new participants) into `full_df` with continuous row numbers `0..12`.
2. `extra_df` has no `bmi` or `cholesterol`, so those cells are `NaN`. **Recalculate `bmi` for the whole of `full_df`** so it has no missing BMIs. Cholesterol stays missing.

In [ ]:
# Your code here
# full_df = 

In [ ]:
# ✅ Run this cell to check B3.6
assert full_df.shape == (13, 9), f"Expected (13, 9), got {full_df.shape}"
assert list(full_df.index) == list(range(13)), "Use ignore_index=True for continuous numbering"
assert full_df["bmi"].isnull().sum() == 0, "Recalculate bmi for all rows"
assert full_df["cholesterol"].isnull().sum() == 4
print("✅ B3.6 passed!")

<details>
<summary>💡 <strong>Hint</strong></summary>

`pd.concat([df1, df2], axis=0, ignore_index=True)`. Columns are matched by **name**, so the column order in `extra_df` doesn't matter.

</details>

<details>
<summary>✅ <strong>Solution</strong></summary>

```python
full_df = pd.concat([left_df, extra_df], axis=0, ignore_index=True)
full_df["bmi"] = full_df["weight"] / full_df["height"] ** 2
full_df
```

</details>

## B4 · Saving and loading (4d)

These exercises write small files into the current folder.

🔁 **Safety net:** if `full_df` isn't right, open the solutions for B3.1–B3.6 and run them in order.

### B4.1 · Tab-separated round trip

1. Save `full_df` as a **tab-separated** file `practice_cohort.tsv`, **without** the index.
2. Read it back into `reloaded`.

In [ ]:
# Your code here
# reloaded = 

In [ ]:
# ✅ Run this cell to check B4.1
with open("practice_cohort.tsv") as f:
    first_line = f.readline()
assert "\t" in first_line and first_line.startswith("participant_id"), "The file should be tab-separated with a header and no index column"
assert reloaded.shape == full_df.shape, f"reloaded should have shape {full_df.shape}, got {reloaded.shape}"
assert list(reloaded.columns) == list(full_df.columns)
print("✅ B4.1 passed!")

<details>
<summary>💡 <strong>Hint</strong></summary>

Use `sep="\t"` for **both** `.to_csv` and `pd.read_csv`, and `index=False` when saving.

</details>

<details>
<summary>✅ <strong>Solution</strong></summary>

```python
full_df.to_csv("practice_cohort.tsv", sep="\t", index=False)
reloaded = pd.read_csv("practice_cohort.tsv", sep="\t")
reloaded.head()
```

</details>

🤔 **Think about it:** run `print(type(reloaded.loc[0, "diseases"]), type(reloaded.loc[0, "age"]))`. What changed after the round trip, and what didn't?

<details>
<summary>👀 <strong>Answer</strong></summary>

`diseases` is now a `str`, while `age` is still a number. Text files only store text. pandas converts columns that look numeric or True/False back automatically, but a tuple like `('Asthma', 'Diabetes')` comes back as the *string* `"('Asthma', 'Diabetes')"`.

</details>

### B4.2 · A file without a header

1. Save `full_df` to `practice_noheader.tsv`: tab-separated, **no header and no index**.
2. Load it into `noheader_df` so that **no data row is lost**.
3. Give `noheader_df` the same column names as `full_df`, without typing them.

In [ ]:
# Your code here
# noheader_df = 

In [ ]:
# ✅ Run this cell to check B4.2
assert noheader_df.shape == full_df.shape, f"Expected {full_df.shape}, got {noheader_df.shape}. Was the first row used as a header?"
assert list(noheader_df.columns) == list(full_df.columns)
assert noheader_df.loc[0, "participant_id"] == "participant1"
print("✅ B4.2 passed!")

<details>
<summary>💡 <strong>Hint</strong></summary>

Saving uses `header=False`. Loading uses `header=None`. Then reuse `full_df.columns`.

</details>

<details>
<summary>✅ <strong>Solution</strong></summary>

```python
full_df.to_csv("practice_noheader.tsv", sep="\t", header=False, index=False)
noheader_df = pd.read_csv("practice_noheader.tsv", sep="\t", header=None)
noheader_df.columns = full_df.columns
noheader_df.head()
```

</details>

### B4.3 · Keeping row labels with `index_col`

1. Save `clinical_df` (row labels `participant1`…) to `practice_indexed.csv`, **keeping** the index.
2. Read it back into `indexed_df` so that the participant names are the **row labels** again and `indexed_df.loc["participant4"]` works.

In [ ]:
# Your code here
# indexed_df = 

In [ ]:
# ✅ Run this cell to check B4.3
assert indexed_df.index[0] == "participant1", "Row labels should be the participant names. Try index_col=0"
assert indexed_df.loc["participant4", "age"] == 83
assert "Unnamed: 0" not in indexed_df.columns
print("✅ B4.3 passed!")

<details>
<summary>💡 <strong>Hint</strong></summary>

`.to_csv` saves the index by default. When reading, `index_col=0` turns the first column back into the index (4c uses this).

</details>

<details>
<summary>✅ <strong>Solution</strong></summary>

```python
clinical_df.to_csv("practice_indexed.csv")
indexed_df = pd.read_csv("practice_indexed.csv", index_col=0)
indexed_df.loc["participant4"]
```

Without `index_col=0` you get an extra column called `Unnamed: 0`, a sign the index was saved as a plain column.

</details>

## B5 · Exploratory data analysis on the stroke dataset (4e)

Run this cell to load the stroke prediction dataset. It looks for the copy in this folder, then in `../data`, then online.

In [ ]:
try:
    stroke = pd.read_csv("healthcare-dataset-stroke-data.csv")
except FileNotFoundError:
    try:
        stroke = pd.read_csv("../data/healthcare-dataset-stroke-data.csv")
    except FileNotFoundError:
        stroke = pd.read_csv("https://raw.githubusercontent.com/hanane-issa/python-crash-course/main/data/healthcare-dataset-stroke-data.csv")
stroke.head()

### B5.1 · First look

1. `n_rows`, `n_cols`: the number of rows and columns. Unpack them from `.shape` in one line.
2. `last_7`: the final 7 rows.
3. `all_but_last_100`: every row **except** the last 100, using `.head`.
4. Also run `stroke.info()` and `stroke.describe()` and read the output. Which column has fewer non-null values than the others?

In [ ]:
# Your code here

In [ ]:
# ✅ Run this cell to check B5.1
assert (n_rows, n_cols) == (5110, 12)
assert len(last_7) == 7 and last_7.index[-1] == 5109
assert len(all_but_last_100) == 5010
print("✅ B5.1 passed!")

<details>
<summary>💡 <strong>Hint</strong></summary>

`a, b = some_tuple` unpacks a tuple. `.head(-n)` drops the last *n* rows.

</details>

<details>
<summary>✅ <strong>Solution</strong></summary>

```python
n_rows, n_cols = stroke.shape
last_7 = stroke.tail(7)
all_but_last_100 = stroke.head(-100)
stroke.info()
stroke.describe()
```

`.info()` shows that `bmi` has 4909 non-null values out of 5110, so 201 are missing.

</details>

### B5.2 · Unique values and aggregation

1. `gender_values`: the unique values in the `gender` column. Is there anything unexpected?
2. `n_strokes`: the number of patients who had a stroke.
3. `binary_summary`: using `.agg`, a table with the **count**, **sum** and **mean** of the columns `stroke`, `hypertension` and `heart_disease`.

In [ ]:
# Your code here

In [ ]:
# ✅ Run this cell to check B5.2
assert set(gender_values) == {"Male", "Female", "Other"}
assert n_strokes == 249, f"Expected 249, got {n_strokes}"
assert binary_summary.shape == (3, 3)
assert binary_summary.loc["sum", "stroke"] == 249
assert abs(binary_summary.loc["mean", "hypertension"] - 0.0975) < 0.001
print("✅ B5.2 passed!")

<details>
<summary>💡 <strong>Hint</strong></summary>

For a column of 0/1 values, `.sum()` is the number of 1s and `.mean()` is the **proportion** of 1s.

</details>

<details>
<summary>✅ <strong>Solution</strong></summary>

```python
gender_values = stroke["gender"].unique()
print(gender_values)
n_strokes = stroke["stroke"].sum()
binary_summary = stroke[["stroke", "hypertension", "heart_disease"]].agg(["count", "sum", "mean"])
binary_summary
```

There is one patient with gender `"Other"`, which is easy to miss without `.unique()`. Note that the `mean` row of `binary_summary` gives the **prevalence** of each condition directly.

</details>

### B5.3 · Filtering rows

1. `older_smokers`: patients aged **60 or over** whose `smoking_status` is `"smokes"`.
2. `hyper_or_heart`: patients with hypertension **or** heart disease.
3. `urban_high_glucose_ages`: the **ages only** (a Series) of urban patients with `avg_glucose_level` above 200. Use `.loc`.

In [ ]:
# Your code here

In [ ]:
# ✅ Run this cell to check B5.3
assert len(older_smokers) == 191, f"Expected 191, got {len(older_smokers)}"
assert len(hyper_or_heart) == 710, f"Expected 710, got {len(hyper_or_heart)}"
assert isinstance(urban_high_glucose_ages, pd.Series), "urban_high_glucose_ages should be a Series of ages"
assert len(urban_high_glucose_ages) == 219, f"Expected 219, got {len(urban_high_glucose_ages)}"
print("✅ B5.3 passed!")

<details>
<summary>💡 <strong>Hint</strong></summary>

Wrap **each** condition in parentheses and join them with `&` or `|`. For column selection: `df.loc[condition, "age"]`.

</details>

<details>
<summary>✅ <strong>Solution</strong></summary>

```python
older_smokers = stroke[(stroke["age"] >= 60) & (stroke["smoking_status"] == "smokes")]
hyper_or_heart = stroke[(stroke["hypertension"] == 1) | (stroke["heart_disease"] == 1)]
urban_high_glucose_ages = stroke.loc[(stroke["Residence_type"] == "Urban") & (stroke["avg_glucose_level"] > 200), "age"]
print(len(older_smokers), len(hyper_or_heart), len(urban_high_glucose_ages))
```

`stroke.loc[condition]["age"]` (the course style) also works for *reading*. The `.loc[condition, "age"]` form is preferred because the same form also works for *assigning*.

</details>

### B5.4 · String filtering with `.str`

1. `smoked_text`: rows where `smoking_status` **contains** `"smoked"`. Look at which statuses that includes. Is it what you'd expect?
2. `former_smokers`: rows where `smoking_status` **starts with** `"formerly"`.
3. `self_employed`: rows where `work_type` contains `"SELF"`, **ignoring case**.
4. `self_employed_former`: rows meeting conditions 2 **and** 3.

In [ ]:
# Your code here

In [ ]:
# ✅ Run this cell to check B5.4
assert len(smoked_text) == 2777, f"Expected 2777, got {len(smoked_text)}"
assert len(former_smokers) == 885
assert len(self_employed) == 819, "Did you use flags=re.IGNORECASE?"
assert len(self_employed_former) == 208
print("✅ B5.4 passed!")

<details>
<summary>💡 <strong>Hint</strong></summary>

`.str.contains(...)` and `.str.startswith(...)` return Boolean Series that you can combine with `&`. For case-insensitive matching use `flags=re.IGNORECASE`.

</details>

<details>
<summary>✅ <strong>Solution</strong></summary>

```python
smoked_text = stroke[stroke["smoking_status"].str.contains("smoked")]
print(smoked_text["smoking_status"].unique())   # includes 'never smoked'!
former_bool = stroke["smoking_status"].str.startswith("formerly")
self_bool = stroke["work_type"].str.contains("SELF", flags=re.IGNORECASE)
former_smokers = stroke[former_bool]
self_employed = stroke[self_bool]
self_employed_former = stroke[former_bool & self_bool]
print(len(smoked_text), len(former_smokers), len(self_employed), len(self_employed_former))
```

⚠️ `"smoked"` also matches **`never smoked`**. Substring searches are easy to get subtly wrong, so always check the matches with `.unique()`.

</details>

### B5.5 · Handling missing data

Work on a copy so `stroke` stays untouched: `clean = stroke.copy()`.

1. `missing_counts`: the number of missing values in each column of `stroke`, using `.isnull()`.
2. In `clean`, replace every `"Unknown"` in `smoking_status` with `pd.NA`.
3. Store the mean BMI (ignoring missing values) in `bmi_mean`, then fill the missing `bmi` values in `clean` with it.

In [ ]:
# Your code here
clean = stroke.copy()

In [ ]:
# ✅ Run this cell to check B5.5
assert missing_counts["bmi"] == 201 and missing_counts["smoking_status"] == 0
assert clean["smoking_status"].isnull().sum() == 1544, "All 1544 'Unknown' values should now be missing"
assert (clean["smoking_status"] == "Unknown").sum() == 0
assert clean["bmi"].isnull().sum() == 0, "There should be no missing BMI values left"
assert abs(bmi_mean - 28.893) < 0.001
assert stroke["bmi"].isnull().sum() == 201, "stroke should be unchanged. Work on clean"
print("✅ B5.5 passed!")

<details>
<summary>💡 <strong>Hint</strong></summary>

Use the pattern `df.loc[condition, "column"] = new_value`. For the BMI, the condition is `clean["bmi"].isnull()`.

</details>

<details>
<summary>✅ <strong>Solution</strong></summary>

```python
clean = stroke.copy()
missing_counts = stroke.isnull().sum()
print(missing_counts)

clean.loc[clean["smoking_status"] == "Unknown", "smoking_status"] = pd.NA

bmi_mean = clean["bmi"].mean(skipna=True)
clean.loc[clean["bmi"].isnull(), "bmi"] = bmi_mean
clean.isnull().sum()
```

Notice that `missing_counts` reports **0** for `smoking_status` even though 1544 values are unknown. pandas only recognises real missing values (`NaN`/`None`/`pd.NA`), not placeholder text.

</details>

---
# Part C: Debugging 🐛

Each cell below contains **broken code** with a stated goal. Some raise an error, and some run but give the **wrong answer**, which is more dangerous. Fix the code in place, then run the check cell.

🔁 **Safety net:** run this cell first.

In [ ]:
clinical_df = pd.DataFrame(participants_clinical_information).T
clinical_df.columns = COLUMNS
clinical_df = clinical_df.infer_objects()
clinical_df.loc["participant1", "height"] = 1.75

### C1 · Building a dataframe (2 bugs)

**Goal:** a dataframe with one row per patient, columns `name`, `age`, `ward`, and row labels `patient_1`…`patient_3`.

In [ ]:
patients = {"name": ["Ana", "Ben", "Cleo"], "age": [34, 51, 29], "ward": ["A", "B", "A"]}
patients_df = pd.dataframe(patients)
patients_df.columns = ["name", "age"]
patients_df.index = ["patient_1", "patient_2", "patient_3"]
patients_df

In [ ]:
# ✅ Run this cell to check C1
assert patients_df.shape == (3, 3)
assert list(patients_df.columns) == ["name", "age", "ward"]
assert patients_df.loc["patient_2", "ward"] == "B"
print("✅ C1 passed!")

<details>
<summary>💡 <strong>Hint</strong></summary>

Read the error message: it names the problem. Fix one bug, re-run, and read the next error.

</details>

<details>
<summary>✅ <strong>Solution</strong></summary>

```python
patients = {"name": ["Ana", "Ben", "Cleo"], "age": [34, 51, 29], "ward": ["A", "B", "A"]}
patients_df = pd.DataFrame(patients)                  # capital D and F
patients_df.columns = ["name", "age", "ward"]         # one label per column
patients_df.index = ["patient_1", "patient_2", "patient_3"]
patients_df
```

</details>

### C2 · Filtering with two conditions

**Goal:** patients **over 50** who **have hypertension**.

In [ ]:
over50_hyp = stroke[stroke["age"] > 50 & stroke["hypertension"] == 1]
len(over50_hyp)

In [ ]:
# ✅ Run this cell to check C2
assert len(over50_hyp) == 403, f"Expected 403, got {len(over50_hyp)}"
print("✅ C2 passed!")

<details>
<summary>💡 <strong>Hint</strong></summary>

Python evaluates `&` **before** `>` and `==`, so without parentheses this is read as `stroke["age"] > (50 & stroke["hypertension"]) == 1`.

</details>

<details>
<summary>✅ <strong>Solution</strong></summary>

```python
over50_hyp = stroke[(stroke["age"] > 50) & (stroke["hypertension"] == 1)]
len(over50_hyp)
```

This is why the course puts every condition in its own parentheses: `(cond1) & (cond2)`.

</details>

### C3 · The silent non-update 🤫

**Goal:** convert participant 1's height from cm to m. This code runs, perhaps with a warning, but **the value doesn't change**. Why? Fix it.

(The same pattern appears in the course's own solution to 4b mini-exercise 3.)

In [ ]:
ht_df = pd.DataFrame(
    {"gender": ["female", "male", "male"], "height": [175.0, 1.79, 1.75]},
    index=["participant1", "participant2", "participant3"],
)
ht_df.loc["participant1"]["height"] = 1.75
ht_df

In [ ]:
# ✅ Run this cell to check C3
assert ht_df.loc["participant1", "height"] == 1.75, "participant1's height is still not 1.75"
print("✅ C3 passed!")

<details>
<summary>💡 <strong>Hint</strong></summary>

`ht_df.loc["participant1"]` creates a **new, temporary Series**. `["height"] = 1.75` then changes that temporary object, not `ht_df`.

</details>

<details>
<summary>✅ <strong>Solution</strong></summary>

```python
ht_df = pd.DataFrame(
    {"gender": ["female", "male", "male"], "height": [175.0, 1.79, 1.75]},
    index=["participant1", "participant2", "participant3"],
)
ht_df.loc["participant1", "height"] = 1.75   # row and column in ONE .loc
ht_df
```

This is called **chained assignment**: two `[]` operations in a row, followed by `=`. For **reading**, `df.loc[row][col]` works fine. For **writing**, always use a single `df.loc[row, col] = value`. In pandas ≥ 3.0 chained assignment *never* works, and older versions only sometimes did, often with a `SettingWithCopyWarning`.

</details>

### C4 · Side-by-side join

**Goal:** a 3-row table with columns `participant` and `score` side by side.

In [ ]:
ids = pd.DataFrame({"participant": ["p1", "p2", "p3"]})
scores = pd.DataFrame({"score": [7, 9, 4]})
combined = pd.concat([ids, scores])
combined

In [ ]:
# ✅ Run this cell to check C4
assert combined.shape == (3, 2), f"Expected (3, 2), got {combined.shape}"
assert combined.isnull().sum().sum() == 0, "There should be no missing values"
print("✅ C4 passed!")

<details>
<summary>💡 <strong>Hint</strong></summary>

What is the default `axis` of `pd.concat`? Look at the output: 6 rows, full of `NaN`.

</details>

<details>
<summary>✅ <strong>Solution</strong></summary>

```python
ids = pd.DataFrame({"participant": ["p1", "p2", "p3"]})
scores = pd.DataFrame({"score": [7, 9, 4]})
combined = pd.concat([ids, scores], axis=1)
combined
```

</details>

### C5 · Merge on keys

**Goal:** add each participant's number of visits to `demo`.

In [ ]:
visits = pd.DataFrame({"ID": ["p2", "p1", "p3"], "visits": [4, 1, 7]})
demo = pd.DataFrame({"participant_name": ["p1", "p2", "p3"], "age": [30, 45, 52]})
merged = demo.merge(visits, left_on="ID", right_on="participant_name")
merged

In [ ]:
# ✅ Run this cell to check C5
assert len(merged) == 3
assert merged.loc[merged["participant_name"] == "p2", "visits"].iloc[0] == 4, "p2 should have 4 visits"
print("✅ C5 passed!")

<details>
<summary>💡 <strong>Hint</strong></summary>

`left_on` names a column in the dataframe **before** `.merge`, and `right_on` names a column in the one passed **into** it.

</details>

<details>
<summary>✅ <strong>Solution</strong></summary>

```python
visits = pd.DataFrame({"ID": ["p2", "p1", "p3"], "visits": [4, 1, 7]})
demo = pd.DataFrame({"participant_name": ["p1", "p2", "p3"], "age": [30, 45, 52]})
merged = demo.merge(visits, left_on="participant_name", right_on="ID")
merged
```

</details>

### C6 · Loading a headerless TSV (2 bugs)

**Goal:** load all three rows of `no_header.tsv` with columns `participant` and `age`. The first line of the cell creates the file and is correct.

In [ ]:
pd.DataFrame([["p1", 25], ["p2", 40], ["p3", 18]]).to_csv("no_header.tsv", sep="\t", header=False, index=False)

loaded = pd.read_csv("no_header.tsv")
loaded.columns = ["participant", "age"]
loaded

In [ ]:
# ✅ Run this cell to check C6
assert loaded.shape == (3, 2), f"Expected (3, 2), got {loaded.shape}"
assert loaded.loc[0, "participant"] == "p1", "The first row was swallowed as a header"
print("✅ C6 passed!")

<details>
<summary>💡 <strong>Hint</strong></summary>

Bug 1: how are the values separated in the file? Bug 2: once that's fixed, count the rows.

</details>

<details>
<summary>✅ <strong>Solution</strong></summary>

```python
pd.DataFrame([["p1", 25], ["p2", 40], ["p3", 18]]).to_csv("no_header.tsv", sep="\t", header=False, index=False)

loaded = pd.read_csv("no_header.tsv", sep="\t", header=None)
loaded.columns = ["participant", "age"]
loaded
```

</details>

### C7 · Finding government workers (2 bugs)

**Goal:** all patients whose `work_type` contains `govt`, ignoring upper/lower case. The actual value in the data is `Govt_job`.

In [ ]:
gov_workers = stroke[stroke[["work_type"]].str.contains("govt")]
len(gov_workers)

In [ ]:
# ✅ Run this cell to check C7
assert len(gov_workers) == 657, f"Expected 657, got {len(gov_workers)}"
print("✅ C7 passed!")

<details>
<summary>💡 <strong>Hint</strong></summary>

`.str` only exists on a **Series**. How many brackets give you a Series? Then: `"govt"` ≠ `"Govt"`.

</details>

<details>
<summary>✅ <strong>Solution</strong></summary>

```python
gov_workers = stroke[stroke["work_type"].str.contains("govt", flags=re.IGNORECASE)]
len(gov_workers)
```

</details>

### C8 · Cleaning missing data that won't clean (2 bugs)

**Goal:** in `debug_df`, turn every `"Unknown"` smoking status into `pd.NA`, and fill missing BMIs with the mean. The code runs without errors but **does nothing**. Find out why.

In [ ]:
debug_df = stroke.copy()
debug_df.loc[debug_df["smoking_status"] == "unknown", "smoking_status"] = pd.NA
bmi_mean = debug_df["bmi"].mean()
debug_df.loc[debug_df["bmi"] == np.nan, "bmi"] = bmi_mean
debug_df.isnull().sum()

In [ ]:
# ✅ Run this cell to check C8
assert debug_df["smoking_status"].isnull().sum() == 1544, "'Unknown' values were not replaced"
assert debug_df["bmi"].isnull().sum() == 0, "Missing BMIs were not filled"
print("✅ C8 passed!")

<details>
<summary>💡 <strong>Hint</strong></summary>

Bug 1 is about capitalisation. For bug 2, try `print(np.nan == np.nan)`.

</details>

<details>
<summary>✅ <strong>Solution</strong></summary>

```python
debug_df = stroke.copy()
debug_df.loc[debug_df["smoking_status"] == "Unknown", "smoking_status"] = pd.NA
bmi_mean = debug_df["bmi"].mean()
debug_df.loc[debug_df["bmi"].isnull(), "bmi"] = bmi_mean
debug_df.isnull().sum()
```

`NaN` is defined to be **not equal to anything, including itself**, so `== np.nan` is always `False`. Always test for missing values with `.isnull()` (or `.isna()`).

</details>

### C9 · Off-by-one with `.iloc`

**Goal:** the **first 4 participants** and the columns `age`, `height` and `weight` (positions 1–3) from `clinical_df`.

In [ ]:
first_four = clinical_df.iloc[1:4, 1:3]
first_four

In [ ]:
# ✅ Run this cell to check C9
assert first_four.shape == (4, 3), f"Expected (4, 3), got {first_four.shape}"
assert first_four.index[0] == "participant1"
assert list(first_four.columns) == ["age", "height", "weight"]
print("✅ C9 passed!")

<details>
<summary>💡 <strong>Hint</strong></summary>

Positions start at **0**, and the end of a slice is **excluded**.

</details>

<details>
<summary>✅ <strong>Solution</strong></summary>

```python
first_four = clinical_df.iloc[0:4, 1:4]
first_four
```

</details>

---
# Part D: Mini-projects 🚀

## D1 · Assemble a multi-site cohort

You're the data manager for a small two-site study. The data arrived in three different shapes:

- **Site A** sent a Python dictionary, one entry per participant (as in 4a).
- **Site B** sent a dataframe with the columns in a different order, a different ID column name (`subject`), and **height in cm**.
- **The lab** sent blood results keyed by `patient_ref`, in a random order. Some participants are missing, one row (`C07`) is for a participant not in the study, and one `hba1c` value is missing.

Run the setup cell, then work through the tasks. You can write your code across as many cells as you like. The final check cell tests everything.

In [ ]:
site_a = {  # participant -> [gender, age, height_m, weight_kg, smoking_status]
    "A01": ["female", 52, 1.64, 70.2, "never"],
    "A02": ["male", 61, 1.78, 88.0, "current"],
    "A03": ["female", 45, 1.70, 65.5, "Unknown"],
    "A04": ["male", 38, 1.83, 79.1, "former"],
    "A05": ["female", 70, 1.58, 59.8, "never"],
}

site_b = pd.DataFrame({
    "weight_kg": [99.5, 58.1, 74.0, 81.6],
    "subject": ["B01", "B02", "B03", "B04"],
    "height_cm": [180.0, 160.0, 172.0, 169.0],
    "age": [57, 33, 66, 49],
    "gender": ["male", "female", "male", "female"],
    "smoking_status": ["former", "never", "Unknown", "current"],
})

bloods = pd.DataFrame({
    "patient_ref": ["B03", "A02", "A05", "B01", "A01", "B04", "A03", "C07"],
    "hba1c": [48.0, 52.0, np.nan, 41.0, 39.0, 44.0, 36.0, 50.0],   # mmol/mol
    "ldl": [3.1, 4.2, 2.8, 3.6, 2.9, 3.3, 2.5, 3.0],               # mmol/L
})

### Tasks

1. **Site A → dataframe.** Build `site_a_df` with one row per participant and columns `gender, age, height_m, weight_kg, smoking_status`. Fix the column types with `.infer_objects()`. Add a `participant_id` column from the row labels, add a `site` column set to `"A"` for every row, and renumber the rows from 0.
2. **Tidy site B.** Create `site_b_df` from `site_b` with a `participant_id` column (copied from `subject`), a `height_m` column (converted from cm), and a `site` column set to `"B"`.
3. **Combine.** Stack both sites into `cohort` (9 rows, continuous index). Keep only the columns `participant_id, site, gender, age, height_m, weight_kg, smoking_status`, in that order.
4. **Add bloods.** Merge the lab results into `cohort`, keeping **every participant** and *only* study participants. Drop the duplicate `patient_ref` column.
5. **BMI.** Add a `bmi` column.
6. **Missing data.** Replace `"Unknown"` smoking statuses with `pd.NA`. Then fill missing `hba1c` values with the mean `hba1c` **of the cohort participants**.
7. **Final layout.** Reorder the columns to `participant_id, site, gender, age, height_m, weight_kg, bmi, smoking_status, hba1c, ldl`.
8. **Save and reload.** Save `cohort` to `cohort_final.tsv` (tab-separated, no index) and read it back into `cohort_reloaded`.
9. **Questions.** Answer these with filtering:
   - `n_high_risk`: the number of participants with BMI ≥ 30 **or** HbA1c ≥ 48.
   - `site_b_ever_smokers`: a dataframe of site B participants whose smoking status is `"former"` **or** `"current"`.

<details>
<summary>💡 <strong>Hints</strong></summary>

- Task 1 is B1.1 + B3.2 again. A single value assigned to a column (`df["site"] = "A"`) is repeated down every row.
- Task 2: `site_b_df = site_b.copy()` first, then add columns. `pd.concat` matches columns **by name**, so the column order doesn't matter, but the **names** do.
- Task 4: which `how=` keeps every row of the left table and nothing extra from the right?
- Task 6: ⚠️ **order matters**. If you take the mean from `bloods` you include `C07`, who isn't in the study. Fill **after** merging, using the mean of `cohort["hba1c"]`.
- Task 9: `(cond1) | (cond2)`, then count the rows with `len(...)`.

</details>

In [ ]:
# Your code here (add more cells if you like)

In [ ]:
# ✅ Run this cell to check D1
expected_cols = ["participant_id", "site", "gender", "age", "height_m", "weight_kg", "bmi", "smoking_status", "hba1c", "ldl"]
assert list(cohort.columns) == expected_cols, f"Columns should be {expected_cols}"
assert cohort.shape == (9, 10), f"Expected 9 participants and 10 columns, got {cohort.shape}"
assert list(cohort.index) == list(range(9)), "Index should be 0..8"
assert sorted(cohort["participant_id"]) == ["A01", "A02", "A03", "A04", "A05", "B01", "B02", "B03", "B04"], "C07 must not be included"
assert list(cohort["site"]) == ["A"] * 5 + ["B"] * 4
assert (cohort["height_m"] < 2.5).all(), "All heights should be in metres"
assert abs(cohort.loc[cohort["participant_id"] == "B01", "bmi"].iloc[0] - 30.71) < 0.01
assert cohort["smoking_status"].isnull().sum() == 2, "The two 'Unknown' statuses should be missing values"
assert cohort["hba1c"].isnull().sum() == 0, "All hba1c values should be filled"
filled = cohort.loc[cohort["participant_id"].isin(["A04", "A05", "B02"]), "hba1c"]
assert (abs(filled - 43.333) < 0.01).all(), "Fill with the mean of the *cohort's* hba1c (43.33), not the lab table's"
assert cohort["ldl"].isnull().sum() == 2, "A04 and B02 have no lab results, so their ldl should stay missing"
assert cohort_reloaded.shape == cohort.shape
assert n_high_risk == 3, f"Expected 3 high-risk participants, got {n_high_risk}"
assert sorted(site_b_ever_smokers["participant_id"]) == ["B01", "B04"]
print("✅ D1 passed! 🎉")

<details>
<summary>✅ <strong>Solution</strong></summary>

```python
# 1. Site A
site_a_df = pd.DataFrame(site_a).T
site_a_df.columns = ["gender", "age", "height_m", "weight_kg", "smoking_status"]
site_a_df = site_a_df.infer_objects()
site_a_df["participant_id"] = site_a_df.index
site_a_df["site"] = "A"
site_a_df.index = np.arange(0, site_a_df.shape[0])

# 2. Site B
site_b_df = site_b.copy()
site_b_df["participant_id"] = site_b_df["subject"]
site_b_df["height_m"] = site_b_df["height_cm"] / 100
site_b_df["site"] = "B"

# 3. Combine
keep = ["participant_id", "site", "gender", "age", "height_m", "weight_kg", "smoking_status"]
cohort = pd.concat([site_a_df[keep], site_b_df[keep]], axis=0, ignore_index=True)

# 4. Bloods (left merge keeps every participant, drops C07)
cohort = cohort.merge(bloods, left_on="participant_id", right_on="patient_ref", how="left")
cohort = cohort.drop(columns="patient_ref")

# 5. BMI
cohort["bmi"] = cohort["weight_kg"] / cohort["height_m"] ** 2

# 6. Missing data (fill AFTER merging so C07 doesn't affect the mean)
cohort.loc[cohort["smoking_status"] == "Unknown", "smoking_status"] = pd.NA
hba1c_mean = cohort["hba1c"].mean(skipna=True)
cohort.loc[cohort["hba1c"].isnull(), "hba1c"] = hba1c_mean

# 7. Final layout
cohort = cohort[["participant_id", "site", "gender", "age", "height_m", "weight_kg",
                 "bmi", "smoking_status", "hba1c", "ldl"]]

# 8. Save and reload
cohort.to_csv("cohort_final.tsv", sep="\t", index=False)
cohort_reloaded = pd.read_csv("cohort_final.tsv", sep="\t")

# 9. Questions
n_high_risk = len(cohort[(cohort["bmi"] >= 30) | (cohort["hba1c"] >= 48)])
site_b_ever_smokers = cohort[(cohort["site"] == "B") &
                             ((cohort["smoking_status"] == "former") | (cohort["smoking_status"] == "current"))]
print("High risk:", n_high_risk)
cohort
```

Note that `ldl` stays missing for A04 and B02. The task only asked to fill `hba1c`. Whether (and how) to fill missing data is a choice you should make deliberately, column by column.

</details>

## D2 · Investigate stroke risk 🔬

Use the original `stroke` dataframe and the `clean` dataframe from B5.5 (re-run the B5.5 solution if needed) to answer these questions. Most need filtering plus `.mean()` on the 0/1 `stroke` column, which gives the **stroke rate**.

1. `overall_rate`: the proportion of all patients who had a stroke.
2. `rate_65_plus` and `rate_under_65`: the stroke rate for patients aged ≥ 65 and < 65. Also `age_ratio`, how many times higher the first is than the second.
3. `rate_hyp` and `rate_no_hyp`: the stroke rate with and without hypertension.
4. `stroke_missing_bmi`: how many **stroke patients** in the *original* `stroke` data have a missing BMI? Why might this matter for an analysis that simply drops rows with missing BMI?
5. **Stroke rate by work type.** Loop over `stroke["work_type"].unique()`. For each work type, filter the rows and record the number of patients, the number of strokes and the stroke rate in lists. Build a dataframe `work_table` with the columns `work_type, n_patients, n_strokes, stroke_rate`. Then use a filter with `.max()` to find `highest_work_type`, the work type with the highest stroke rate.
6. **Smoking table.** Using `clean`, build `smoking_table` the same way for the statuses `["never smoked", "formerly smoked", "smokes"]`, and include a `mean_age` column too.

In [ ]:
# Your code here (add more cells if you like)

In [ ]:
# ✅ Run this cell to check D2
assert abs(overall_rate - 0.0487) < 0.0001
assert abs(rate_65_plus - 0.1548) < 0.0001 and abs(rate_under_65 - 0.0220) < 0.0001
assert abs(age_ratio - 7.02) < 0.01
assert abs(rate_hyp - 0.1325) < 0.0001 and abs(rate_no_hyp - 0.0397) < 0.0001
assert stroke_missing_bmi == 40
assert list(work_table.columns) == ["work_type", "n_patients", "n_strokes", "stroke_rate"]
assert len(work_table) == 5 and work_table["n_patients"].sum() == 5110
assert highest_work_type == "Self-employed"
assert list(smoking_table.columns) == ["smoking_status", "n_patients", "n_strokes", "stroke_rate", "mean_age"]
assert list(smoking_table["n_strokes"]) == [90, 70, 42]
print("✅ D2 passed! 🎉")

<details>
<summary>✅ <strong>Solution</strong></summary>

```python
# 1
overall_rate = stroke["stroke"].mean()

# 2
rate_65_plus = stroke[stroke["age"] >= 65]["stroke"].mean()
rate_under_65 = stroke[stroke["age"] < 65]["stroke"].mean()
age_ratio = rate_65_plus / rate_under_65

# 3
rate_hyp = stroke[stroke["hypertension"] == 1]["stroke"].mean()
rate_no_hyp = stroke[stroke["hypertension"] == 0]["stroke"].mean()

# 4
stroke_missing_bmi = stroke[stroke["stroke"] == 1]["bmi"].isnull().sum()

# 5
work_types, n_patients, n_strokes, rates = [], [], [], []
for work in stroke["work_type"].unique():
    subset = stroke[stroke["work_type"] == work]
    work_types.append(work)
    n_patients.append(len(subset))
    n_strokes.append(subset["stroke"].sum())
    rates.append(subset["stroke"].mean())
work_table = pd.DataFrame({"work_type": work_types, "n_patients": n_patients,
                           "n_strokes": n_strokes, "stroke_rate": rates})
highest_work_type = work_table.loc[work_table["stroke_rate"] == work_table["stroke_rate"].max(), "work_type"].iloc[0]

# 6
statuses = ["never smoked", "formerly smoked", "smokes"]
n_patients, n_strokes, rates, ages = [], [], [], []
for status in statuses:
    subset = clean[clean["smoking_status"] == status]
    n_patients.append(len(subset))
    n_strokes.append(subset["stroke"].sum())
    rates.append(subset["stroke"].mean())
    ages.append(subset["age"].mean())
smoking_table = pd.DataFrame({"smoking_status": statuses, "n_patients": n_patients,
                              "n_strokes": n_strokes, "stroke_rate": rates, "mean_age": ages})
print(work_table)
smoking_table
```

**Q4:** 40 of the 249 stroke patients (16%) have no BMI, compared with about 3% of non-stroke patients. Dropping rows with missing BMI would remove stroke cases unevenly and bias the results.

</details>

### 🤔 Reflection questions (no code check)

1. **Former** smokers have a higher stroke rate (7.9%) than **current** smokers (5.3%). Does that mean quitting smoking raises your stroke risk? Look at the `mean_age` column of `smoking_table`.
2. `Self-employed` has the highest stroke rate of the work types. What other variable might explain that? Try checking the mean age per work type.
3. The `children` work type has a stroke rate close to 0. Why should you be careful about including children when comparing groups of adults?

<details>
<summary>👀 <strong>Discussion</strong></summary>

All three are about **confounding by age**. Former smokers are on average about 8 years older (55 vs 47), self-employed patients are older than private-sector workers, and children are, well, children. Age is by far the strongest stroke risk factor in this dataset (about 7× higher rate at 65+), so comparisons between groups with different age profiles can mislead. A fairer comparison would look *within* age bands, for example by repeating the smoking table only for patients aged 50–70.

</details>

---
# ✅ Self-assessment checklist

Tick these off when you can do them **without looking anything up**:

- [ ] Create a dataframe from a dictionary or NumPy array, and know when to transpose (B1)
- [ ] Set and read `.columns` and `.index` (B1, B3.2)
- [ ] Explain the difference between `df["a"]` and `df[["a"]]` (A5, B2.1)
- [ ] Select with `.loc` (labels, inclusive) and `.iloc` (positions, end-exclusive) (B2.2, B2.3, C9)
- [ ] Change values safely with a single `df.loc[row, col] = ...` and avoid chained assignment (B2.4, C3)
- [ ] Add rows and columns, and reorder columns (B3.1–B3.3)
- [ ] Choose between `pd.concat` (by position/index) and `.merge` (by key) (B3.4, B3.5, C4, C5)
- [ ] Save and load CSV/TSV with `sep`, `header`, `index` and `index_col` (B4, C6)
- [ ] Explore data with `.head/.tail/.info/.describe/.unique/.agg` (B5.1, B5.2)
- [ ] Filter with `&`/`|` and parentheses, and with `.str.contains/.startswith` (B5.3, B5.4, C2, C7)
- [ ] Detect and handle missing data with `.isnull()`, `pd.NA` and mean filling (B5.5, C8)